## 2.1 一阶马尔可夫模型 + 拉普拉斯平滑

序列："ababc"，词汇表 V = {a, b, c}，|V| = 3

---

### 统计转移：

序列划分为转移对：a→b, b→a, a→b, b→c

b 出现次数 count(b) = 2
count(b→a) = 1
count(b→c) = 1
count(b→b) = 0

---

### 拉普拉斯平滑公式：

p(x | 'b') = (count(b→x) + 1) / (count(b) + |V|)

---

### 1. p('a' | 'b')

p('a' | 'b') = (1 + 1) / (2 + 3) = 2/5 = 0.4

---

### 2. p('c' | 'b')

p('c' | 'b') = (1 + 1) / (2 + 3) = 2/5 = 0.4

---

注：p('b' | 'b') = (0 + 1) / (2 + 3) = 1/5 = 0.2
三者之和 = 0.4 + 0.4 + 0.2 = 1.0，满足概率归一化

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理，生成 n-gram 特征和标签
    
    参数:
    - text: 输入文本字符串
    - n: 滑动窗口大小
    
    返回:
    - vocab: 词汇表字典 {词: id}
    - (features, labels): 特征列表和标签列表
    """
    # 1. 转小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按频率排序，ID从0开始）
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.keys(), key=lambda w: word_counts[w], reverse=True)
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    
    for i in range(len(words) - n):
        # 取 n 个词作为特征
        feature = words[i:i+n]
        # 下一个词作为标签
        label = words[i+n]
        features.append(feature)
        labels.append(label)
    
    return vocab, (features, labels)


# ============================================
# 测试
# ============================================
print("=" * 60)
print("2.2 文本预处理测试")
print("=" * 60)

# 测试1：题目示例
print("\n测试1：'The time machine'，n=2")
text1 = "The time machine"
vocab1, (feat1, lab1) = preprocess_text(text1, 2)
print(f"词汇表: {vocab1}")
print(f"特征: {feat1}")
print(f"标签: {lab1}")

# 测试2：更复杂的文本
print("\n测试2：'The quick brown fox jumps over the lazy dog'，n=3")
text2 = "The quick brown fox jumps over the lazy dog"
vocab2, (feat2, lab2) = preprocess_text(text2, 3)
print(f"词汇表: {vocab2}")
print(f"特征数量: {len(feat2)}")
for i in range(min(3, len(feat2))):
    print(f"  特征 {i+1}: {feat2[i]} -> 标签: {lab2[i]}")

# 测试3：带标点符号的文本
print("\n测试3：带标点文本 'Hello, World! How are you?'，n=2")
text3 = "Hello, World! How are you?"
vocab3, (feat3, lab3) = preprocess_text(text3, 2)
print(f"词汇表: {vocab3}")
print(f"特征: {feat3}")
print(f"标签: {lab3}")

# 测试4：边界情况（文本长度不足）
print("\n测试4：短文本 'hi'，n=2")
text4 = "hi"
vocab4, (feat4, lab4) = preprocess_text(text4, 2)
print(f"词汇表: {vocab4}")
print(f"特征: {feat4}")
print(f"标签: {lab4}")
print("（文本长度不足 n+1，特征和标签均为空）")

print("\n" + "=" * 60)
print("测试完成！")

2.2 文本预处理测试

测试1：'The time machine'，n=2
词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征: [['the', 'time']]
标签: ['machine']

测试2：'The quick brown fox jumps over the lazy dog'，n=3
词汇表: {'the': 0, 'quick': 1, 'brown': 2, 'fox': 3, 'jumps': 4, 'over': 5, 'lazy': 6, 'dog': 7}
特征数量: 6
  特征 1: ['the', 'quick', 'brown'] -> 标签: fox
  特征 2: ['quick', 'brown', 'fox'] -> 标签: jumps
  特征 3: ['brown', 'fox', 'jumps'] -> 标签: over

测试3：带标点文本 'Hello, World! How are you?'，n=2
词汇表: {'hello': 0, 'world': 1, 'how': 2, 'are': 3, 'you': 4}
特征: [['hello', 'world'], ['world', 'how'], ['how', 'are']]
标签: ['how', 'are', 'you']

测试4：短文本 'hi'，n=2
词汇表: {'hi': 0}
特征: []
标签: []
（文本长度不足 n+1，特征和标签均为空）

测试完成！


## 3.1 线性RNN梯度推导（BPTT）

已知：
ht = Whh·h_{t-1} + Whx·xt
ot = Woh·ht
L = (1/2) · Σ_{t=1}^T (ot - yt)^2

---

### 推导 ∂L/∂Whh：

设 Lt = (1/2)(ot - yt)^2

由链式法则：
∂Lt/∂Whh = (∂Lt/∂ot) · (∂ot/∂ht) · (∂ht/∂Whh)

其中：
∂Lt/∂ot = ot - yt
∂ot/∂ht = Woh^T
∂ht/∂Whh 需要递归展开

---

ht 对 Whh 的依赖包含直接和间接两部分：
ht = Whh·h_{t-1} + Whx·xt

其中 h_{t-1} 本身也依赖 Whh，因此：
∂ht/∂Whh = h_{t-1}^T + Whh · (∂h_{t-1}/∂Whh)

递归展开至 h_0：
∂ht/∂Whh = Σ_{i=1}^t (Π_{k=i+1}^t Whh) · h_{i-1}^T

---

最终：
∂L/∂Whh = Σ_{t=1}^T (ot - yt)^T · Woh^T · Σ_{i=1}^t (Π_{k=i+1}^t Whh) · h_{i-1}^T

---

### 梯度消失/爆炸的条件：

连乘项 Π_{k=i+1}^t Whh 决定了梯度的传播行为。

设 Whh 的特征值分解，最大特征值为 λ_max：

- 梯度爆炸条件：|λ_max| > 1
  连乘项指数增长，梯度随 t-i 增大而爆炸

- 梯度消失条件：|λ_max| < 1
  连乘项指数衰减，早期时间步对梯度贡献趋近于零

- 梯度稳定条件：|λ_max| = 1（实践中难以实现）

---

结论：线性RNN的梯度传播完全由 Whh 的特征值决定。
     这是原始RNN难以学习长距离依赖的根本原因，
     也催生了LSTM/GRU等门控架构。

In [2]:
import numpy as np

def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单步前向传播
    
    参数:
    - x_t: 当前输入，形状 (batch_size, input_size)
    - h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
    - W_hx: 输入到隐藏权重，形状 (input_size, hidden_size)
    - W_hh: 隐藏到隐藏权重，形状 (hidden_size, hidden_size)
    - b_h: 隐藏层偏置，形状 (hidden_size,)
    
    返回:
    - h_t: 当前隐藏状态
    - cache: 缓存中间值用于反向传播
    """
    # 线性变换
    a_t = x_t @ W_hx + h_prev @ W_hh + b_h
    
    # tanh 激活
    h_t = np.tanh(a_t)
    
    # 缓存
    cache = (x_t, h_prev, W_hx, W_hh, a_t, h_t)
    
    return h_t, cache


def rnn_step_backward(dh_next, cache):
    """
    RNN 单步反向传播
    
    参数:
    - dh_next: 损失对 h_t 的梯度，形状 (batch_size, hidden_size)
    - cache: 前向传播缓存
    
    返回:
    - dx_t: 损失对 x_t 的梯度
    - dh_prev: 损失对 h_prev 的梯度
    - dW_hx: 损失对 W_hx 的梯度
    - dW_hh: 损失对 W_hh 的梯度
    - db_h: 损失对 b_h 的梯度
    """
    x_t, h_prev, W_hx, W_hh, a_t, h_t = cache
    
    # tanh 的导数：1 - tanh(a)^2 = 1 - h_t^2
    da_t = dh_next * (1 - h_t ** 2)
    
    # 计算各梯度
    dx_t = da_t @ W_hx.T                    # (batch, input_size)
    dh_prev = da_t @ W_hh.T                 # (batch, hidden_size)
    dW_hx = x_t.T @ da_t                    # (input_size, hidden_size)
    dW_hh = h_prev.T @ da_t                 # (hidden_size, hidden_size)
    db_h = np.sum(da_t, axis=0)             # (hidden_size,)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ============================================
# 测试
# ============================================
print("=" * 60)
print("3.2 RNN 单元前向+反向传播测试")
print("=" * 60)

# 设置参数
np.random.seed(42)
batch_size = 3
input_size = 4
hidden_size = 5

# 随机初始化
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b_h = np.random.randn(hidden_size) * 0.1

print(f"\n输入 x_t 形状: {x_t.shape}")
print(f"隐藏状态 h_prev 形状: {h_prev.shape}")
print(f"权重 W_hx 形状: {W_hx.shape}")
print(f"权重 W_hh 形状: {W_hh.shape}")
print(f"偏置 b_h 形状: {b_h.shape}")

# 前向传播
h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
print(f"\n前向传播输出 h_t 形状: {h_t.shape}")
print(f"h_t 值范围: [{h_t.min():.4f}, {h_t.max():.4f}]")

# 反向传播
dh_next = np.random.randn(batch_size, hidden_size) * 0.1
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

print(f"\n反向传播梯度：")
print(f"  dx_t 形状: {dx_t.shape}")
print(f"  dh_prev 形状: {dh_prev.shape}")
print(f"  dW_hx 形状: {dW_hx.shape}")
print(f"  dW_hh 形状: {dW_hh.shape}")
print(f"  db_h 形状: {db_h.shape}")

# 梯度数值验证
print(f"\n梯度数值验证（与 PyTorch 对比）：")
try:
    import torch
    
    # PyTorch 版本
    x_t_torch = torch.tensor(x_t, requires_grad=True)
    h_prev_torch = torch.tensor(h_prev, requires_grad=True)
    W_hx_torch = torch.tensor(W_hx, requires_grad=True)
    W_hh_torch = torch.tensor(W_hh, requires_grad=True)
    b_h_torch = torch.tensor(b_h, requires_grad=True)
    
    a_t_torch = x_t_torch @ W_hx_torch + h_prev_torch @ W_hh_torch + b_h_torch
    h_t_torch = torch.tanh(a_t_torch)
    
    dh_next_torch = torch.tensor(dh_next)
    h_t_torch.backward(dh_next_torch)
    
    # 对比
    print(f"  dW_hx 最大差异: {np.abs(dW_hx - W_hx_torch.grad.numpy()).max():.8f}")
    print(f"  dW_hh 最大差异: {np.abs(dW_hh - W_hh_torch.grad.numpy()).max():.8f}")
    print(f"  db_h 最大差异: {np.abs(db_h - b_h_torch.grad.numpy()).max():.8f}")
    print(f"  dx_t 最大差异: {np.abs(dx_t - x_t_torch.grad.numpy()).max():.8f}")
    print(f"  dh_prev 最大差异: {np.abs(dh_prev - h_prev_torch.grad.numpy()).max():.8f}")
    print(f"\n  梯度计算正确！")
    
except ImportError:
    print("  未安装 PyTorch，跳过对比验证")

print("\n" + "=" * 60)
print("测试完成！")

3.2 RNN 单元前向+反向传播测试

输入 x_t 形状: (3, 4)
隐藏状态 h_prev 形状: (3, 5)
权重 W_hx 形状: (4, 5)
权重 W_hh 形状: (5, 5)
偏置 b_h 形状: (5,)

前向传播输出 h_t 形状: (3, 5)
h_t 值范围: [-0.7108, 0.3541]

反向传播梯度：
  dx_t 形状: (3, 4)
  dh_prev 形状: (3, 5)
  dW_hx 形状: (4, 5)
  dW_hh 形状: (5, 5)
  db_h 形状: (5,)

梯度数值验证（与 PyTorch 对比）：
  dW_hx 最大差异: 0.00000000
  dW_hh 最大差异: 0.00000000
  db_h 最大差异: 0.00000000
  dx_t 最大差异: 0.00000000
  dh_prev 最大差异: 0.00000000

  梯度计算正确！

测试完成！


## 4.1 深度双向RNN参数量计算

已知：L层，每层隐藏单元数H，输入维度D，输出维度O

---

### 参数分析：

每层包含一个前向RNN和一个后向RNN，每个RNN的参数包括：
- W_hx（输入到隐藏权重）
- W_hh（隐藏到隐藏权重）
- b_h（偏置）

---

### 第1层：

输入维度为 D

前向RNN参数：
  W_hx: D × H
  W_hh: H × H
  b_h: H
  小计：D×H + H×H + H = H(D + H + 1)

后向RNN参数（与前向相同）：
  小计：H(D + H + 1)

第1层总参数：2H(D + H + 1)

---

### 第2到L层（共 L-1 层）：

由于双向拼接，每层的输入维度变为 2H（上一层的前向+后向拼接）

单层前向RNN参数：
  W_hx: 2H × H
  W_hh: H × H
  b_h: H
  小计：2H×H + H×H + H = H(3H + 1)

后向RNN参数相同：H(3H + 1)

单层总参数：2H(3H + 1)

L-1层总参数：(L-1) × 2H(3H + 1)

---

### 总参数量（不含输出层）：

总参数 = 第1层 + 第2~L层
       = 2H(D + H + 1) + (L-1) × 2H(3H + 1)
       = 2H(D + H + 1 + (L-1)(3H + 1))

展开：
总参数 = 2H[D + H + 1 + 3H(L-1) + (L-1)]
       = 2H[D + H + 1 + 3HL - 3H + L - 1]
       = 2H[D - 2H + 3HL + L]
       = 2H[D + H(3L - 2) + L]

---

答案：总参数量 = 2H[D + H(3L - 2) + L]
      （不包括输出层的 O × 2H + O 参数）

In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    """
    双向RNN编码器
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BiRNNEncoder, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 双向RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False  # 输入形状 (seq_len, batch, input_dim)
        )
    
    def forward(self, X):
        """
        参数:
        - X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
        - all_hidden: 每个时间步的拼接隐藏状态 (seq_len, batch, 2*hidden_dim)
        - final_hidden: 最终时间步的拼接隐藏状态 (batch, 2*hidden_dim)
        """
        # RNN 前向传播
        # output: (seq_len, batch, 2*hidden_dim)
        # h_n: (2*num_layers, batch, hidden_dim)
        output, h_n = self.rnn(X)
        
        # 获取最终时间步的隐藏状态
        # h_n 形状: (2*num_layers, batch, hidden_dim)
        # 取最后一层的前向和后向
        if self.num_layers == 1:
            h_forward = h_n[0]   # (batch, hidden_dim)
            h_backward = h_n[1]  # (batch, hidden_dim)
        else:
            # 多层时，取最后一层的前向和后向
            h_forward = h_n[-2]  # 倒数第二个（最后一层前向）
            h_backward = h_n[-1]  # 最后一个（最后一层后向）
        
        # 拼接前向和后向
        final_hidden = torch.cat([h_forward, h_backward], dim=1)  # (batch, 2*hidden_dim)
        
        return output, final_hidden


# ============================================
# 测试
# ============================================
print("=" * 60)
print("4.2 双向RNN编码器测试")
print("=" * 60)

# 参数设置
seq_len = 5
batch_size = 3
input_dim = 10
hidden_dim = 8
num_layers = 2

# 创建编码器
encoder = BiRNNEncoder(input_dim, hidden_dim, num_layers)
print(f"\n编码器结构:\n{encoder}")

# 生成随机输入
torch.manual_seed(42)
X = torch.randn(seq_len, batch_size, input_dim)
print(f"\n输入 X 形状: {X.shape}")

# 前向传播
all_hidden, final_hidden = encoder(X)

print(f"\n所有时间步隐藏状态形状: {all_hidden.shape}")
print(f"  (seq_len={seq_len}, batch={batch_size}, 2*hidden_dim={2*hidden_dim})")
print(f"\n最终时间步隐藏状态形状: {final_hidden.shape}")
print(f"  (batch={batch_size}, 2*hidden_dim={2*hidden_dim})")

# 验证
print(f"\n验证：")
print(f"  all_hidden 最后一个时间步 == final_hidden 的前向部分？")
print(f"  all_hidden[-1, :, :hidden_dim] 与 h_forward 一致: ", end="")
last_forward = all_hidden[-1, :, :hidden_dim]
print(torch.allclose(last_forward, final_hidden[:, :hidden_dim], atol=1e-6))

# 总参数量
total_params = sum(p.numel() for p in encoder.parameters())
print(f"\n总参数量: {total_params:,}")

print("\n" + "=" * 60)
print("测试完成！")

4.2 双向RNN编码器测试

编码器结构:
BiRNNEncoder(
  (rnn): RNN(10, 8, num_layers=2, bidirectional=True)
)

输入 X 形状: torch.Size([5, 3, 10])

所有时间步隐藏状态形状: torch.Size([5, 3, 16])
  (seq_len=5, batch=3, 2*hidden_dim=16)

最终时间步隐藏状态形状: torch.Size([3, 16])
  (batch=3, 2*hidden_dim=16)

验证：
  all_hidden 最后一个时间步 == final_hidden 的前向部分？
  all_hidden[-1, :, :hidden_dim] 与 h_forward 一致: True

总参数量: 736

测试完成！


## 5.1 Skip-gram 负采样损失函数

给定中心词 w_c，上下文词 w_o，K 个负样本 {w_n1, w_n2, ..., w_nK}

---

### 目标函数（最大化对数似然）：

对于一对中心词和上下文词 (w_c, w_o)，目标是：
- 正样本 (w_c, w_o) 的预测概率尽可能大
- K 个负样本 (w_c, w_nk) 的预测概率尽可能小

定义正样本概率（sigmoid）：
P(D=1 | w_c, w_o) = σ(v_c · u_o) = 1 / (1 + exp(-v_c · u_o))

负样本概率：
P(D=0 | w_c, w_nk) = σ(-v_c · u_nk) = 1 - σ(v_c · u_nk)

---

### 一对样本的损失函数（负对数似然）：

L = -[log σ(v_c · u_o) + Σ_{k=1}^K log σ(-v_c · u_nk)]

其中：
- v_c：中心词 w_c 的词向量
- u_o：正样本上下文词 w_o 的词向量
- u_nk：第 k 个负样本词的词向量
- σ(x) = 1/(1+e^{-x})

---

### 完整目标函数（整个语料库）：

J = Σ_{(w_c, w_o) ∈ D} [log σ(v_c · u_o) + Σ_{k=1}^K E_{w_n ~ P_n(w)} log σ(-v_c · u_n)]

其中 D 是所有正样本对的集合。

---

### 负样本采样方式：

噪声分布 P_n(w) 通常使用：
P_n(w) = (freq(w))^(3/4) / Σ_w' (freq(w'))^(3/4)

其中 freq(w) 是词 w 在语料中的频率，3/4 次幂可以提升低频词的采样概率。

经验上这个分布也被称为 Unigram 分布的 3/4 次幂平滑。

---

答案：
L(w_c, w_o, {w_nk}) = -[log σ(v_c · u_o) + Σ_{k=1}^K log σ(-v_c · u_nk)]

In [4]:
import numpy as np

def cbow_forward(context_indices, target_idx, W, W_out):
    """
    CBOW 模型前向传播 + 交叉熵损失计算
    
    参数:
    - context_indices: 上下文词索引，形状 (batch_size, context_size)
    - target_idx: 目标中心词索引，形状 (batch_size,)
    - W: 输入权重矩阵（词嵌入），形状 (V, d)
    - W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
    - loss: 交叉熵损失值
    - probs: softmax 输出概率分布，形状 (batch_size, V)
    """
    batch_size = context_indices.shape[0]
    
    # 1. 查嵌入表，获取上下文词的词向量
    # context_indices: (batch_size, context_size)
    # 取出的向量: (batch_size, context_size, d)
    context_vectors = W[context_indices]  # (batch_size, context_size, d)
    
    # 2. 计算平均上下文向量（隐藏层）
    h = np.mean(context_vectors, axis=1)  # (batch_size, d)
    
    # 3. 计算输出层得分
    scores = h @ W_out  # (batch_size, V)
    
    # 4. Softmax 计算概率分布
    # 减去最大值防止数值溢出
    scores_max = np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    # 5. 交叉熵损失
    # 取目标词对应位置的预测概率
    batch_indices = np.arange(batch_size)
    target_probs = probs[batch_indices, target_idx]
    
    # 防止 log(0)
    target_probs = np.clip(target_probs, 1e-15, 1.0)
    loss = -np.mean(np.log(target_probs))
    
    return loss, probs, h


# ============================================
# 测试
# ============================================
print("=" * 60)
print("5.2 CBOW 模型测试")
print("=" * 60)

# 参数设置
np.random.seed(42)
V = 10       # 词汇表大小
d = 5        # 嵌入维度
batch_size = 4
context_size = 3

# 随机初始化权重
W = np.random.randn(V, d) * 0.1       # 输入权重 (V, d)
W_out = np.random.randn(d, V) * 0.1   # 输出权重 (d, V)

# 随机生成数据
context_indices = np.random.randint(0, V, (batch_size, context_size))
target_idx = np.random.randint(0, V, (batch_size,))

print(f"\n词汇表大小 V = {V}")
print(f"嵌入维度 d = {d}")
print(f"批大小 = {batch_size}, 上下文窗口 = {context_size}")
print(f"\n上下文词索引:\n{context_indices}")
print(f"目标词索引: {target_idx}")

# 前向传播
loss, probs, h = cbow_forward(context_indices, target_idx, W, W_out)

print(f"\n隐藏层 h 形状: {h.shape}  (batch_size={batch_size}, d={d})")
print(f"输出概率分布形状: {probs.shape}  (batch_size={batch_size}, V={V})")
print(f"\n交叉熵损失: {loss:.6f}")

# 验证概率和为1
print(f"\n每行概率和: {probs.sum(axis=1)}")
print(f"所有概率 >= 0: {np.all(probs >= 0)}")

# 对比随机猜测的损失
random_loss = -np.log(1.0 / V)
print(f"\n随机猜测的期望损失 (-log(1/V)): {random_loss:.6f}")
print(f"模型损失: {loss:.6f}")
print(f"（模型损失应接近随机损失，因为权重是随机初始化的）")

print("\n" + "=" * 60)
print("测试完成！")

5.2 CBOW 模型测试

词汇表大小 V = 10
嵌入维度 d = 5
批大小 = 4, 上下文窗口 = 3

上下文词索引:
[[8 4 0]
 [2 9 7]
 [5 7 8]
 [3 0 0]]
目标词索引: [9 3 6 1]

隐藏层 h 形状: (4, 5)  (batch_size=4, d=5)
输出概率分布形状: (4, 10)  (batch_size=4, V=10)

交叉熵损失: 2.301799

每行概率和: [1. 1. 1. 1.]
所有概率 >= 0: True

随机猜测的期望损失 (-log(1/V)): 2.302585
模型损失: 2.301799
（模型损失应接近随机损失，因为权重是随机初始化的）

测试完成！


## 6.1 缩放点积注意力计算

已知：
Q ∈ R^(2×4)，K ∈ R^(3×4)，V ∈ R^(3×5)，dk = 4

---

### 第1步：计算得分矩阵 S = Q·K^T / √dk

K^T 形状: (4, 3)
S = Q @ K^T 形状: (2, 3)

设 Q = [q1; q2]（两行），K = [k1; k2; k3]（三行）

S_ij = (qi · kj) / √4 = (qi · kj) / 2

得分矩阵 S 形状为 (2, 3)，表示2个查询对3个键的相似度。

---

### 第2步：对得分矩阵逐行做 Softmax

A = softmax(S, dim=1)

A_ij = exp(S_ij) / Σ_{j'=1}^3 exp(S_ij')

注意力权重矩阵 A 形状: (2, 3)
每行和为 1。

---

### 第3步：加权求和

输出 = A @ V

输出形状: (2, 3) @ (3, 5) = (2, 5)

每个查询得到一个 5 维的输出向量，是 3 个值向量的加权和。

---

### 总结：

输入：Q(2×4), K(3×4), V(3×5)
中间：S = QK^T/√4 → (2×3)
      A = softmax(S) → (2×3)
输出：A @ V → (2×5)

缩放因子 √dk = 2 的作用：
防止点积值过大导致 softmax 梯度趋近于零（饱和区），
使注意力权重分布更加平滑。